# 02_mean_baseline

После EDA логично начать с самого простого и понятного ориентира.
Нам нужен baseline, который легко интерпретируется и сразу показывает,
насколько трудной оказывается задача прогнозирования в этой постановке.

Здесь мы строим mean-baseline в двух вариантах:
на всех рядах и отдельно на наиболее значимых сериях. Это позволяет использовать
одну и ту же простую модель и как общий нижний ориентир, и как честную точку сравнения
для более сильных подходов.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'
METADATA_DIR = ARTIFACTS_DIR / 'metadata'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
SUBMISSIONS_DIR = ARTIFACTS_DIR / 'submissions'

for path in [METRICS_DIR, SUBMISSIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_processed = pd.read_csv(DATASETS_DIR / 'train_processed.csv', parse_dates=['date'])
test_processed = pd.read_csv(DATASETS_DIR / 'test_processed.csv', parse_dates=['date'])
zero_sales = pd.read_csv(DATASETS_DIR / 'zero_sales.csv')
top_pairs_df = pd.read_csv(DATASETS_DIR / 'top_pairs.csv')

with open(METADATA_DIR / 'splits.json', 'r', encoding='utf-8') as f:
    splits_json = json.load(f)

splits = []
for s in splits_json:
    splits.append({
        'name': s['name'],
        'train_idx': pd.Index(s['train_idx']),
        'val_idx': pd.Index(s['val_idx']),
        'train_end': pd.to_datetime(s['train_end']),
        'val_start': pd.to_datetime(s['val_start']),
        'val_end': pd.to_datetime(s['val_end']),
    })

top_pairs = set(zip(top_pairs_df['store_nbr'], top_pairs_df['family']))
zero_pairs = set(zip(zero_sales['store_nbr'], zero_sales['family']))


## Базовая модель и протокол оценки

Идея baseline здесь намеренно простая.
Сначала используем максимально специфичную среднюю для комбинации магазин-товар-день недели,
а если таких наблюдений недостаточно, постепенно переходим к более грубым уровням усреднения.

Такая конструкция хороша тем, что остается прозрачной,
но при этом уже учитывает базовую недельную структуру спроса.


In [ ]:
def filter_to_top_pairs(df, top_pairs):
    mask = df[['store_nbr', 'family']].apply(tuple, axis=1).isin(top_pairs)
    return df[mask].copy()


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    denom = np.sum(np.abs(y_true))
    wape = np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE': wape}


def fit_mean_mappings(train_df):
    df = train_df.copy()
    df['day_of_week'] = df['date'].dt.dayofweek.astype('int8')
    df['log_sales'] = np.log1p(df['sales'].clip(lower=0))
    g_sfd = df.groupby(['store_nbr', 'family', 'day_of_week'])['log_sales'].mean().reset_index(name='log_mean_sfd')
    g_fd = df.groupby(['family', 'day_of_week'])['log_sales'].mean().reset_index(name='log_mean_fd')
    g_f = df.groupby(['family'])['log_sales'].mean().reset_index(name='log_mean_f')
    return {'g_sfd': g_sfd, 'g_fd': g_fd, 'g_f': g_f}


def predict_mean_baseline(target_df, mappings, zero_pairs=None):
    df = target_df.copy()
    df['day_of_week'] = df['date'].dt.dayofweek.astype('int8')
    df = df[['store_nbr', 'family', 'day_of_week']].copy()
    df = df.merge(mappings['g_sfd'], on=['store_nbr', 'family', 'day_of_week'], how='left')
    df = df.merge(mappings['g_fd'], on=['family', 'day_of_week'], how='left')
    df = df.merge(mappings['g_f'], on=['family'], how='left')

    log_pred = df['log_mean_sfd'].copy()
    mask = log_pred.isna()
    log_pred.loc[mask] = df.loc[mask, 'log_mean_fd']
    mask = log_pred.isna()
    log_pred.loc[mask] = df.loc[mask, 'log_mean_f']
    log_pred = log_pred.fillna(0.0)

    y_pred = np.expm1(log_pred.to_numpy())
    y_pred = np.clip(y_pred, 0, None)

    if zero_pairs is not None:
        mask_zero = target_df[['store_nbr', 'family']].apply(tuple, axis=1).isin(zero_pairs)
        y_pred[mask_zero.to_numpy()] = 0.0

    return y_pred


In [ ]:
def evaluate_scope(scope_name, use_top_pairs):
    rows = []
    for split in splits:
        train_df = train_processed.loc[split['train_idx']].copy()
        val_df = train_processed.loc[split['val_idx']].copy()
        if use_top_pairs:
            train_df = filter_to_top_pairs(train_df, top_pairs)
            val_df = filter_to_top_pairs(val_df, top_pairs)
        mappings = fit_mean_mappings(train_df)
        y_pred = predict_mean_baseline(val_df, mappings, zero_pairs=zero_pairs)
        metrics = calculate_metrics(val_df['sales'].to_numpy(), y_pred)
        metrics.update({'split': split['name'], 'scope': scope_name, 'model': 'mean_baseline'})
        rows.append(metrics)
    return pd.DataFrame(rows)

metrics_all = evaluate_scope('all_series', use_top_pairs=False)
metrics_top = evaluate_scope('top_pairs', use_top_pairs=True)
metrics_all


In [ ]:
metrics_top


In [ ]:
metrics_all.to_csv(METRICS_DIR / 'mean_baseline_metrics_all_series.csv', index=False)
metrics_top.to_csv(METRICS_DIR / 'mean_baseline_metrics_top_pairs.csv', index=False)
summary = (
    pd.concat([metrics_all, metrics_top], ignore_index=True)
    .groupby(['model', 'scope'], as_index=False)[['RMSLE', 'RMSE', 'MAE', 'WAPE']]
    .mean()
)
summary.to_csv(METRICS_DIR / 'mean_baseline_metrics_summary.csv', index=False)
summary


## Финальный прогноз для тестового горизонта

После проверки на временных сплитах ту же самую baseline-модель можно применить
ко всему обучающему периоду и получить простой внешний прогноз.

Этот шаг полезен не столько сам по себе, сколько как ориентир:
после него будет понятно, насколько сильно более сложные модели действительно улучшают результат.


In [ ]:
full_mappings = fit_mean_mappings(train_processed)
test_preds = predict_mean_baseline(test_processed, full_mappings, zero_pairs=zero_pairs)
submission = pd.DataFrame({'id': test_processed['id'].values, 'sales': np.clip(test_preds, 0, None)})
submission.to_csv(SUBMISSIONS_DIR / 'submission_mean_baseline.csv', index=False)
submission.head()
